# 03 — Volatility Forecasting

This notebook compares simple and parametric volatility forecasts and produces the monthly volatility input used by the portfolio backtest.

## 0. Setup and helper functions


In [153]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model


In [154]:
default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

In [155]:
returns = pd.read_csv("data/returns.csv", index_col=0, parse_dates=True)

IS_END = "2017-12-31"
OOS_START = "2018-01-01"
HORIZON = 21
TRADING_DAYS = 252
CASH_PROXY = "SHY"
risk_model_assets = [asset for asset in returns.columns if asset != CASH_PROXY]

OUTPUT_DIR = Path("results/volatility_forecast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [156]:
def evaluate_vol_forecast(vol_forecast, realized_vol, eps=1e-8):
    """Compare volatility forecasts with the forward realized-volatility target."""
    common_index = vol_forecast.index.intersection(realized_vol.index)
    common_cols = vol_forecast.columns.intersection(realized_vol.columns)

    forecast = vol_forecast.loc[common_index, common_cols]
    realized = realized_vol.loc[common_index, common_cols]

    error = forecast - realized

    forecast_var = forecast.pow(2).clip(lower=eps)
    realized_var = realized.pow(2).clip(lower=eps)

    ratio = realized_var / forecast_var

    qlike = (
        ratio
        - np.log(ratio)
        - 1
    ).mean()

    summary = pd.DataFrame({
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "Bias": error.mean(),
        "Correlation": forecast.corrwith(realized),
        "QLIKE": qlike
    })

    return summary.round(4)


In [157]:
def vol_forecast_regression_hac(vol_forecast, realized_vol, maxlags=21):
    """
    Run asset-level calibration regressions with HAC standard errors.

        realized_vol_t = alpha + beta * forecast_vol_t + error_t

    HAC errors are used because the 21-day realized-volatility target
    overlaps across adjacent forecast dates.
    """

    common_index = vol_forecast.index.intersection(realized_vol.index)
    common_cols = vol_forecast.columns.intersection(realized_vol.columns)

    results = []

    for asset in common_cols:
        data = pd.DataFrame({
            "realized": realized_vol.loc[common_index, asset],
            "forecast": vol_forecast.loc[common_index, asset]
        }).dropna()

        y = data["realized"]
        X = sm.add_constant(data["forecast"])

        model = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": maxlags}
        )

        # Joint calibration check: zero intercept and unit forecast loading.
        joint_test = model.f_test("const = 0, forecast = 1")

        results.append({
            "Asset": asset,
            "Alpha": model.params["const"],
            "Beta": model.params["forecast"],
            "Alpha p-value": model.pvalues["const"],
            "Beta p-value": model.pvalues["forecast"],
            "R2": model.rsquared,
            "Joint F-stat": float(joint_test.fvalue),
            "Joint p-value": float(joint_test.pvalue),
        })

    return pd.DataFrame(results).set_index("Asset").round(4)


In [158]:
def run_arch_lm_tests(returns_data, lags=5):
    """Run asset-level ARCH-LM tests on demeaned returns."""

    results = []

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        # Demean returns before testing for ARCH effects.
        residuals = r - r.mean()

        lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(
            residuals,
            nlags=lags
        )

        results.append({
            "Asset": asset,
            "ARCH-LM Stat": lm_stat,
            "ARCH-LM p-value": lm_pvalue,
            "F-stat": f_stat,
            "F-test p-value": f_pvalue,
            "Lags": lags,
            "N obs": len(r)
        })

    return pd.DataFrame(results).set_index("Asset").round(4)


In [159]:
def fit_garch(returns_data, vol="GARCH", p=1, o=0, q=1, dist="normal"):
    """Fit a univariate GARCH-family model for each asset and collect diagnostics."""

    models = {}
    results = []

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        model = arch_model(
            r,
            mean="Constant",
            vol=vol,
            p=p,
            o=o,
            q=q,
            dist=dist,
            rescale=False
        )

        fitted = model.fit(disp="off")
        models[asset] = fitted

        params = fitted.params
        alpha = params.get("alpha[1]", np.nan)
        gamma = params.get("gamma[1]", np.nan)
        beta = params.get("beta[1]", np.nan)
        convergence_flag = getattr(fitted, "convergence_flag", np.nan)
        converged = bool(convergence_flag == 0) if pd.notna(convergence_flag) else np.nan

        row = {
            "Asset": asset,
            "Omega": params.get("omega", np.nan),
            "Alpha": alpha,
        }

        if o == 1:
            row.update({
                "Gamma": gamma,
                "Beta": beta,
                "Persistence": alpha + 0.5 * gamma + beta,
                "Gamma p-value": fitted.pvalues.get("gamma[1]", np.nan),
            })
        else:
            row.update({
                "Beta": beta,
                "Alpha + Beta": alpha + beta,
            })

        row.update({
            "Nu": params.get("nu", np.nan),
            "Converged": converged,
            "LogLik": fitted.loglikelihood,
            "AIC": fitted.aic,
            "BIC": fitted.bic
        })

        results.append(row)

    results = pd.DataFrame(results).set_index("Asset").round(4)

    return models, results


In [160]:
def forecast_sanity_table(vol_forecast):
    """Summarize the distribution of forecast volatilities by asset."""
    return pd.DataFrame({
        "Min": vol_forecast.min(),
        "Median": vol_forecast.median(),
        "Mean": vol_forecast.mean(),
        "Max": vol_forecast.max()
    }).round(4)


In [161]:
def garch_oos_forecast(
    returns_data,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t",
    is_end="2017-12-31",
    oos_start="2018-01-01",
    horizon=21
):
    """Generate OOS annualized volatility forecasts from fixed GARCH parameters."""
    forecasts = {}
    fitted_models = {}

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        model = arch_model(
            r,
            mean="Constant",
            vol=model_type,
            p=p,
            o=o,
            q=q,
            dist=dist,
            rescale=False
        )

        fitted = model.fit(last_obs=is_end, disp="off")
        fitted_models[asset] = fitted

        forecast = fitted.forecast(
            start=oos_start,
            horizon=horizon,
            reindex=True,
            align="origin"
        )

        # Average expected daily variance across the forecast horizon.
        avg_var_forecast = forecast.variance.mean(axis=1)

        # Convert percent daily variance to annualized decimal volatility.
        forecasts[asset] = np.sqrt(avg_var_forecast) / 100 * np.sqrt(TRADING_DAYS)

    forecasts = pd.DataFrame(forecasts).loc[oos_start:]

    return forecasts.dropna(how="all"), fitted_models


## 1. Realized Volatility Target and Rolling Baseline

Forecasts are evaluated against a forward 21-trading-day realized-volatility target. For asset $i$ and forecast date $t$, the target uses returns from $t+1$ to $t+21$:

$$
RV_{i,t:t+21} = \sum_{j=1}^{21} r_{i,t+j}^{2}.
$$

The realized variance is converted to annualized realized volatility as:

$$
\sigma^{\mathrm{realized}}_{i,t:t+21}
=
\sqrt{\frac{252}{21} RV_{i,t:t+21}}.
$$

This target is ex-post and is used only for evaluation. Forecast inputs remain dated at $t$ or earlier; the target measures what is subsequently realized. Keeping these two objects separate avoids look-ahead in model construction and makes the timing of the evaluation explicit.


In [162]:
realized_var_21d = (
    returns.pow(2)
    .rolling(window=HORIZON)
    .sum()
    .shift(-HORIZON)
)

realized_vol_21d = np.sqrt(realized_var_21d * TRADING_DAYS / HORIZON)

is_index = returns.loc[:IS_END].index
IS_EVAL_END = is_index[-HORIZON - 1]

realized_vol_is = realized_vol_21d.loc[:IS_EVAL_END]
realized_vol_oos = realized_vol_21d.loc[OOS_START:].dropna(how="all")


### Rolling Historical Volatility Baseline

The first benchmark is a 63-day rolling standard deviation of daily returns, annualized with 252 trading days. It is included as a transparent baseline for the model-selection exercise.

The estimate is shifted by one day, so the forecast at date $t$ uses only returns observed up to $t-1$. The same daily annualized volatility estimate is evaluated against the forward 21-day realized target, which treats the current risk estimate as the expected average risk level over the next month.

This benchmark is simple and stable, but it has two predictable drawbacks: it weights every observation in the window equally, and it reacts only after large moves enter the rolling window.


In [163]:
ROLLING_WINDOW = 63

In [164]:
rolling_vol = returns.rolling(window=ROLLING_WINDOW).std() * np.sqrt(TRADING_DAYS)
rolling_vol_lagged = rolling_vol.shift(1)
rolling_vol_lagged.tail()

,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-06-02,0.143461,0.221285,0.311249,0.054386,0.100566,0.068035,0.056529,0.275429,0.247440,0.148392,0.015696
2026-06-03,0.141872,0.212134,0.292856,0.054402,0.100660,0.068031,0.056371,0.261981,0.247452,0.148296,0.015695
2026-06-04,0.142557,0.211525,0.294097,0.054474,0.100764,0.068208,0.056273,0.261708,0.247199,0.148296,0.015679
2026-06-05,0.141867,0.207053,0.290527,0.054264,0.100647,0.067907,0.055763,0.261762,0.247999,0.151089,0.015679
2026-06-08,0.149354,0.212925,0.320712,0.055203,0.100879,0.068723,0.055815,0.268261,0.241880,0.149886,0.016193


In [165]:
asset = "SPY"

vol_compare = pd.DataFrame({
    "Rolling 63D Forecast": rolling_vol_lagged[asset].loc[:IS_EVAL_END],
    "Forward 21D Realized Vol": realized_vol_is[asset]
}).dropna()

In [166]:
fig = go.Figure()

for col in vol_compare.columns:
    fig.add_trace(
        go.Scatter(
            x=vol_compare.index,
            y=vol_compare[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset}: Rolling Volatility Forecast vs Forward Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    **default_layout
)

fig.show()

The SPY chart is shown for the in-sample period to keep the visual check separate from OOS evaluation. The rolling forecast captures the main volatility regimes, including 2008-2009 and the sovereign-credit stress period, but it reacts with a delay and remains elevated after shocks fade. That lag is the main weakness of a fixed-window estimator.


### Error Metrics

Forecasts are evaluated with MAE, RMSE, bias, forecast-realized correlation, and QLIKE. MAE and RMSE summarize error in volatility units, bias shows average over- or under-forecasting, correlation measures timing, and QLIKE evaluates forecasts in variance space. Lower MAE, RMSE, absolute bias, and QLIKE are preferred; higher correlation is preferred.


In [167]:
evaluate_vol_forecast(rolling_vol_lagged.loc[:IS_EVAL_END], realized_vol_is)

,MAE,RMSE,Bias,Correlation,QLIKE
SPY,0.0570,0.0964,0.0071,0.6563,0.3873
EFA,0.0661,0.1149,0.0082,0.6392,0.3705
EEM,0.0853,0.1601,0.0100,0.6494,0.3217
IEF,0.0142,0.0188,0.0013,0.6587,0.1401
TLT,0.0293,0.0415,0.0023,0.6020,0.1554
LQD,0.0218,0.0593,0.0029,0.4199,0.6155
HYG,0.0385,0.0712,0.0051,0.6421,0.8271
GLD,0.0440,0.0652,0.0042,0.6287,0.2620
DBC,0.0414,0.0555,0.0026,0.7730,0.1862
VNQ,0.0822,0.1425,0.0097,0.8181,0.3141


The rolling benchmark carries useful signal: correlations are positive for every asset. Errors are lowest for short-duration fixed income and highest for assets exposed to larger regime shifts, such as EEM, VNQ, and SPY.

The positive bias indicates mild over-forecasting on average. For allocation, that is less concerning than persistent under-forecasting, but the delayed response around stress periods leaves room for a more adaptive benchmark.


## 2. EWMA / RiskMetrics Volatility Forecast

EWMA is added as a second benchmark. It keeps the historical-volatility logic but gives more weight to recent squared returns, allowing forecasts to update faster after market moves.

I use $\lambda = 0.94$, the standard daily RiskMetrics setting. The forecast is shifted by one day, so the date-$t$ estimate uses information available before $t$.


In [168]:
LAMBDA_EWMA = 0.94

ewma_var = returns.pow(2).ewm(alpha=1 - LAMBDA_EWMA, adjust=False).mean()
ewma_vol = np.sqrt(ewma_var) * np.sqrt(TRADING_DAYS)

ewma_vol_lagged = ewma_vol.shift(1)

In [169]:
asset = "SPY"

vol_compare_spy = pd.DataFrame({
    "Rolling 63D Forecast": rolling_vol_lagged[asset].loc[:IS_EVAL_END],
    "Forward 21D Realized Vol": realized_vol_is[asset],
    "EWMA Forecast": ewma_vol_lagged[asset].loc[:IS_EVAL_END]
}).dropna()

fig = go.Figure()

for col in vol_compare_spy.columns:
    fig.add_trace(
        go.Scatter(
            x=vol_compare_spy.index,
            y=vol_compare_spy[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset}: Volatility Forecasts vs Forward Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    **default_layout
)

fig.show()

In [170]:
evaluate_vol_forecast(ewma_vol_lagged.loc[:IS_EVAL_END], realized_vol_is)

,MAE,RMSE,Bias,Correlation,QLIKE
SPY,0.0512,0.0831,0.0033,0.7452,0.3633
EFA,0.0588,0.0991,0.0041,0.7331,0.3261
EEM,0.0739,0.1306,0.0053,0.7652,0.2784
IEF,0.0140,0.0189,0.0004,0.6722,0.1535
TLT,0.0286,0.0405,0.0005,0.6470,0.3001
LQD,0.0204,0.0550,0.0015,0.4995,0.6437
HYG,0.0351,0.0659,0.0024,0.6951,0.9439
GLD,0.0414,0.0609,0.0019,0.6841,0.2643
DBC,0.0398,0.0530,0.0016,0.7970,0.1639
VNQ,0.0721,0.1198,0.0042,0.8692,0.2889


For SPY, EWMA responds faster than the 63-day rolling estimate around 2008-2009. It still does not fully match the timing or size of realized volatility spikes, but it reduces the fixed-window lag.

The cross-asset table shows the same trade-off. EWMA improves responsiveness while remaining a benchmark rather than a final model: errors are still material for higher-volatility assets, and calibration has to be checked directly.


### Forecast Validation Regression

I use asset-level calibration regressions to check whether the forecast is informative and whether its scale is aligned with realized volatility:

$$
\sigma^{\mathrm{realized}}_{i,t:t+21}
=
\alpha_i
+
\beta_i \hat{\sigma}_{i,t}
+u_{i,t}.
$$

A positive $\beta_i$ indicates timing information. The joint test $\alpha_i = 0$ and $\beta_i = 1$ is a stricter calibration check. HAC/Newey-West standard errors are used because the 21-day realized target overlaps across adjacent dates.


In [171]:
ewma_regression = vol_forecast_regression_hac(
    vol_forecast=ewma_vol_lagged.loc[:IS_EVAL_END],
    realized_vol=realized_vol_is,
    maxlags=HORIZON
)

ewma_regression

,Alpha,Beta,Alpha p-value,Beta p-value,R2,Joint F-stat,Joint p-value
Asset,,,,,,,
SPY,0.0341,0.7758,0.0147,0.0000,0.5553,3.0135,0.0493
EFA,0.0439,0.7653,0.0118,0.0000,0.5374,3.2058,0.0407
EEM,0.0500,0.7914,0.0910,0.0000,0.5856,1.5096,0.2212
IEF,0.0186,0.7143,0.0000,0.0000,0.4519,12.3819,0.0000
TLT,0.0447,0.6807,0.0000,0.0000,0.4186,20.6128,0.0000
LQD,0.0307,0.5164,0.0000,0.0002,0.2495,8.7194,0.0002
HYG,0.0237,0.7174,0.0028,0.0000,0.4831,4.5639,0.0105
GLD,0.0467,0.7240,0.0000,0.0000,0.4680,8.9182,0.0001
DBC,0.0290,0.8317,0.0025,0.0000,0.6352,4.5596,0.0105


EWMA is informative in-sample: beta is positive and statistically significant for every asset. Explanatory power is strongest for VNQ, SHY, DBC, EEM, and EFA, and weaker for LQD and TLT.

The joint calibration test is rejected for most assets, so EWMA should not be treated as unbiased in level. It is a useful adaptive benchmark, but the fixed decay parameter leaves no room for asset-specific shock sensitivity or persistence.


## 3. GARCH Model Selection

The next step is to estimate asset-level GARCH specifications. The goal is not to overfit each ETF, but to test whether a common, interpretable volatility model is justified by the in-sample return dynamics.

Before fitting GARCH, I check for ARCH effects in returns. This is a diagnostic step: if returns show volatility clustering, conditional-volatility models are more defensible than a fixed-window estimate alone.


### ARCH-LM Test

The ARCH-LM test is used as a pre-check for volatility clustering. For each asset, demeaned returns are used to test whether lagged squared residuals explain current squared residuals:

$$
\hat{\varepsilon}_{i,t}^{2}
=
\alpha_0
+
\sum_{k=1}^{q} \alpha_k \hat{\varepsilon}_{i,t-k}^{2}
+u_{i,t}.
$$

The null is no ARCH effects. Rejection supports conditional-volatility modelling for that asset.


In [172]:
returns_pct = returns.mul(100).dropna(how="all")
returns_pct_is = returns_pct.loc[:IS_END]
returns_pct_oos = returns_pct.loc[OOS_START:]

In [173]:
arch_lm_results_5 = run_arch_lm_tests(
    returns_data=returns_pct_is,
    lags=5
)

arch_lm_results_5

,ARCH-LM Stat,ARCH-LM p-value,F-stat,F-test p-value,Lags,N obs
Asset,,,,,,
SPY,639.7688,0.0,167.3915,0.0,5,2701
EFA,547.9064,0.0,137.2257,0.0,5,2701
EEM,715.4175,0.0,194.3340,0.0,5,2701
IEF,116.1993,0.0,24.2326,0.0,5,2701
TLT,238.0508,0.0,52.1049,0.0,5,2701
LQD,1011.5096,0.0,323.0604,0.0,5,2701
HYG,341.3643,0.0,77.9968,0.0,5,2701
GLD,103.9196,0.0,21.5691,0.0,5,2701
DBC,404.4464,0.0,94.9540,0.0,5,2701


The ARCH-LM results reject the no-ARCH null across the universe. This supports including conditional-volatility models in the comparison rather than relying only on rolling historical volatility.


In [174]:
arch_lm_results_10 = run_arch_lm_tests(
    returns_data=returns_pct_is,
    lags=10
)

arch_lm_results_10

,ARCH-LM Stat,ARCH-LM p-value,F-stat,F-test p-value,Lags,N obs
Asset,,,,,,
SPY,737.8618,0.0,101.2458,0.0,10,2701
EFA,704.0840,0.0,94.9685,0.0,10,2701
EEM,1016.9378,0.0,162.8012,0.0,10,2701
IEF,155.6509,0.0,16.4531,0.0,10,2701
TLT,276.4605,0.0,30.6855,0.0,10,2701
LQD,1454.2311,0.0,315.1227,0.0,10,2701
HYG,692.7435,0.0,92.9086,0.0,10,2701
GLD,170.4416,0.0,18.1223,0.0,10,2701
DBC,490.4573,0.0,59.7319,0.0,10,2701


### Baseline Gaussian GARCH(1,1)

The baseline specification is Gaussian GARCH(1,1). It provides the reference point for the Student-t and GJR extensions:

$$
r_{i,t} = \mu_i + \varepsilon_{i,t},
\qquad
\varepsilon_{i,t} = \sigma_{i,t} z_{i,t}
$$

$$
\sigma_{i,t}^{2}
=
\omega_i
+
\alpha_i \varepsilon_{i,t-1}^{2}
+
\beta_i \sigma_{i,t-1}^{2}.
$$

For model selection, the key diagnostics are shock sensitivity ($\alpha$), persistence ($\beta$), total persistence ($\alpha + \beta$), likelihood-based criteria, and convergence behavior.


In [175]:
garch_norm_models, garch_norm_results = fit_garch(returns_pct_is)

garch_norm_results

,Omega,Alpha,Beta,Alpha + Beta,Nu,Converged,LogLik,AIC,BIC
Asset,,,,,,,,,
SPY,0.0202,0.1287,0.8578,0.9865,NaN,True,-3634.6251,7277.2501,7300.8556
EFA,0.0119,0.1126,0.8874,1.0000,NaN,True,-4201.7419,8411.4839,8435.0894
EEM,0.0247,0.0976,0.8967,0.9944,NaN,True,-4887.2577,9782.5155,9806.1210
IEF,0.0013,0.0411,0.9525,0.9936,NaN,True,-1427.8181,2863.6362,2887.2417
TLT,0.0109,0.0466,0.9408,0.9874,NaN,True,-3486.1627,6980.3255,7003.9310
LQD,0.0063,0.1391,0.8364,0.9755,NaN,True,-1280.3762,2568.7525,2592.3580
HYG,0.0042,0.1543,0.8457,1.0000,NaN,True,-1861.3567,3730.7135,3754.3190
GLD,0.0126,0.0502,0.9414,0.9915,NaN,True,-4043.8493,8095.6985,8119.3040
DBC,0.0041,0.0427,0.9552,0.9978,NaN,True,-4062.7745,8133.5491,8157.1546


The Gaussian GARCH estimates show high persistence across the universe, with $alpha + beta$ close to one for most assets. That confirms that volatility shocks decay slowly and that a conditional-volatility model is appropriate.

The baseline is useful, but not final. Several assets sit very close to the persistence boundary, and Gaussian innovations may understate tail risk. That motivates testing Student-t innovations next.


### Student-t GARCH(1,1)

Student-t innovations are tested to allow fatter residual tails while keeping the same GARCH(1,1) variance recursion:

$$
\varepsilon_{i,t} = \sigma_{i,t} z_{i,t},
\qquad
z_{i,t} \sim t_{\nu_i}.
$$

This is a specification check: if information criteria improve without creating unstable estimates, the distributional assumption is better aligned with the return data.


In [176]:
garch_t_models, garch_t_results = fit_garch(returns_pct_is, vol="GARCH", p=1, o=0, q=1, dist="t")

garch_t_results

,Omega,Alpha,Beta,Alpha + Beta,Nu,Converged,LogLik,AIC,BIC
Asset,,,,,,,,,
SPY,0.0132,0.1330,0.8670,1.0000,5.3218,True,-3564.4169,7138.8337,7168.3406
EFA,0.0105,0.1038,0.8962,1.0000,6.8996,True,-4151.7158,8313.4316,8342.9385
EEM,0.0240,0.0902,0.9036,0.9938,9.8859,True,-4864.8411,9739.6822,9769.1891
IEF,0.0010,0.0418,0.9532,0.9950,13.8964,True,-1410.1016,2830.2032,2859.7101
TLT,0.0078,0.0437,0.9475,0.9912,15.2177,True,-3472.5918,6955.1836,6984.6905
LQD,0.0026,0.0589,0.9245,0.9835,7.0928,True,-1165.0640,2340.1281,2369.6350
HYG,0.0035,0.1524,0.8476,1.0000,4.5707,True,-1691.1146,3392.2292,3421.7361
GLD,0.0073,0.0411,0.9544,0.9955,5.3872,True,-3945.1944,7900.3887,7929.8956
DBC,0.0031,0.0461,0.9529,0.9991,9.1667,True,-4029.3371,8068.6742,8098.1811


The Student-t model raises a convergence warning for SHY, and its persistence is slightly above one. The information-criterion values for SHY are therefore not interpretable.

This is not a portfolio-wide failure. SHY is a low-volatility cash proxy, and the additional tail parameter is poorly identified for this series. I keep the Student-t comparison for the risk assets and treat SHY separately in the final forecast construction.


In [177]:
garch_dist_comparison = pd.DataFrame({
    "Gaussian AIC": garch_norm_results["AIC"],
    "Student-t AIC": garch_t_results["AIC"],
    "AIC Improvement": garch_norm_results["AIC"] - garch_t_results["AIC"],
    "Gaussian BIC": garch_norm_results["BIC"],
    "Student-t BIC": garch_t_results["BIC"],
    "BIC Improvement": garch_norm_results["BIC"] - garch_t_results["BIC"],
}).round(4)

garch_dist_comparison.loc[risk_model_assets]

,Gaussian AIC,Student-t AIC,AIC Improvement,Gaussian BIC,Student-t BIC,BIC Improvement
Asset,,,,,,
SPY,7277.2501,7138.8337,138.4164,7300.8556,7168.3406,132.5150
EFA,8411.4839,8313.4316,98.0523,8435.0894,8342.9385,92.1509
EEM,9782.5155,9739.6822,42.8333,9806.1210,9769.1891,36.9319
IEF,2863.6362,2830.2032,33.4330,2887.2417,2859.7101,27.5316
TLT,6980.3255,6955.1836,25.1419,7003.9310,6984.6905,19.2405
LQD,2568.7525,2340.1281,228.6244,2592.3580,2369.6350,222.7230
HYG,3730.7135,3392.2292,338.4843,3754.3190,3421.7361,332.5829
GLD,8095.6985,7900.3887,195.3098,8119.3040,7929.8956,189.4084
DBC,8133.5491,8068.6742,64.8749,8157.1546,8098.1811,58.9735


Excluding SHY, the information-criterion comparison favors Student-t innovations for every asset. The improvement is strongest for HYG, GLD, LQD, SPY, and EFA, which supports using a fat-tailed innovation distribution for the risk assets.

The conclusion should be stated carefully: Student-t improves in-sample likelihood fit, but it still has to prove useful in the out-of-sample forecast comparison.


### Testing Asymmetric Volatility Responses

The final candidate is GJR-GARCH with Student-t innovations. It tests whether negative shocks have a different impact on next-period volatility than positive shocks of the same size, which is most relevant for equity and credit assets:

$$
\sigma_{i,t}^{2}
=
\omega_i
+
\alpha_i \varepsilon_{i,t-1}^{2}
+
\gamma_i \varepsilon_{i,t-1}^{2} I(\varepsilon_{i,t-1}<0)
+
\beta_i \sigma_{i,t-1}^{2}.
$$

A positive $\gamma_i$ means negative shocks raise conditional volatility more than positive shocks of the same magnitude. The question is whether that extra parameter improves the model enough to justify using it in portfolio inputs.


In [178]:
gjr_t_models, gjr_t_results = fit_garch(returns_pct_is, vol="GARCH", p=1, o=1, q=1, dist="t")

gjr_t_results

,Omega,Alpha,Gamma,Beta,Persistence,Gamma p-value,Nu,Converged,LogLik,AIC,BIC
Asset,,,,,,,,,,,
SPY,0.0172,0.0000,0.2565,0.8647,0.9929,0.0000,5.7325,True,-3510.5863,7033.1726,7068.5809
EFA,0.0112,0.0126,0.1487,0.9085,0.9954,0.0000,7.0914,True,-4123.4187,8258.8373,8294.2456
EEM,0.0182,0.0105,0.1162,0.9253,0.9940,0.0000,10.9838,True,-4841.0513,9694.1026,9729.5108
IEF,0.0010,0.0450,-0.0075,0.9537,0.9949,0.4663,13.8454,True,-1409.8505,2831.7010,2867.1093
TLT,0.0068,0.0537,-0.0273,0.9529,0.9929,0.0117,15.5706,True,-3469.6287,6951.2575,6986.6657
LQD,0.0025,0.0378,0.0292,0.9302,0.9826,0.0578,7.1804,True,-1163.3067,2338.6134,2374.0217
HYG,0.0025,0.0367,0.1810,0.8728,1.0000,0.0000,4.7765,True,-1655.6846,3323.3693,3358.7775
GLD,0.0074,0.0503,-0.0185,0.9549,0.9959,0.0901,5.3749,True,-3943.7916,7899.5832,7934.9915
DBC,0.0025,0.0301,0.0272,0.9555,0.9992,0.0118,9.2537,True,-4025.8929,8063.7858,8099.1941


SHY again produces unstable GARCH estimates. I exclude SHY from the economic interpretation of the GJR comparison and keep it assigned to EWMA in the final construction.


In [179]:
asymmetry_comparison = pd.DataFrame({
    "GARCH-t AIC": garch_t_results["AIC"],
    "GJR-t AIC": gjr_t_results["AIC"],
    "AIC Improvement": garch_t_results["AIC"] - gjr_t_results["AIC"],
    "GARCH-t BIC": garch_t_results["BIC"],
    "GJR-t BIC": gjr_t_results["BIC"],
    "BIC Improvement": garch_t_results["BIC"] - gjr_t_results["BIC"],
    "Gamma": gjr_t_results["Gamma"],
    "Gamma p-value": gjr_t_results["Gamma p-value"]
}).round(4)

asymmetry_comparison.loc[risk_model_assets]

,GARCH-t AIC,GJR-t AIC,AIC Improvement,GARCH-t BIC,GJR-t BIC,BIC Improvement,Gamma,Gamma p-value
Asset,,,,,,,,
SPY,7138.8337,7033.1726,105.6611,7168.3406,7068.5809,99.7597,0.2565,0.0000
EFA,8313.4316,8258.8373,54.5943,8342.9385,8294.2456,48.6929,0.1487,0.0000
EEM,9739.6822,9694.1026,45.5796,9769.1891,9729.5108,39.6783,0.1162,0.0000
IEF,2830.2032,2831.7010,-1.4978,2859.7101,2867.1093,-7.3992,-0.0075,0.4663
TLT,6955.1836,6951.2575,3.9261,6984.6905,6986.6657,-1.9752,-0.0273,0.0117
LQD,2340.1281,2338.6134,1.5147,2369.6350,2374.0217,-4.3867,0.0292,0.0578
HYG,3392.2292,3323.3693,68.8599,3421.7361,3358.7775,62.9586,0.1810,0.0000
GLD,7900.3887,7899.5832,0.8055,7929.8956,7934.9915,-5.0959,-0.0185,0.0901
DBC,8068.6742,8063.7858,4.8884,8098.1811,8099.1941,-1.0130,0.0272,0.0118


The GJR results support asymmetry for the equity-sensitive assets and HYG: SPY, EEM, EFA, VNQ, and HYG have positive significant gamma estimates and better information criteria than symmetric Student-t GARCH.

The evidence is weaker for defensive and diversifying assets. IEF does not benefit from the asymmetric term, and DBC/TLT show statistical significance with limited information-criterion improvement. GLD has a small negative gamma, so the standard leverage interpretation does not apply cleanly.

This argues against selecting GJR automatically for the whole universe. It may fit some assets better in sample, but the final decision should be based on OOS forecast quality and robustness.


### Out-of-Sample Evaluation

The candidate GARCH specifications are evaluated on post-2018 forecasts against the same 21-day forward realized-volatility target. Parameters are estimated through the in-sample cutoff. These tables are validation diagnostics; they should not be described as a second round of strategy tuning.


In [180]:
garch_t_oos_vol, garch_t_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [181]:
gjr_t_oos_vol, gjr_t_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=1,
    q=1,
    dist="t",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [182]:
garch_norm_oos_vol, garch_norm_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="normal",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [183]:
garch_norm_oos_eval = evaluate_vol_forecast(
    vol_forecast=garch_norm_oos_vol,
    realized_vol=realized_vol_oos
)

garch_t_oos_eval = evaluate_vol_forecast(
    vol_forecast=garch_t_oos_vol,
    realized_vol=realized_vol_oos
)

gjr_t_oos_eval = evaluate_vol_forecast(
    vol_forecast=gjr_t_oos_vol,
    realized_vol=realized_vol_oos
)

### Forecast Error Metrics


In [184]:
garch_oos_comparison = pd.DataFrame({
    "GARCH-N MAE": garch_norm_oos_eval["MAE"],
    "GARCH-t MAE": garch_t_oos_eval["MAE"],
    "GJR-t MAE": gjr_t_oos_eval["MAE"],

    "GARCH-N RMSE": garch_norm_oos_eval["RMSE"],
    "GARCH-t RMSE": garch_t_oos_eval["RMSE"],
    "GJR-t RMSE": gjr_t_oos_eval["RMSE"],

    "GARCH-N Corr": garch_norm_oos_eval["Correlation"],
    "GARCH-t Corr": garch_t_oos_eval["Correlation"],
    "GJR-t Corr": gjr_t_oos_eval["Correlation"],

    "GARCH-N QLIKE": garch_norm_oos_eval["QLIKE"],
    "GARCH-t QLIKE": garch_t_oos_eval["QLIKE"],
    "GJR-t QLIKE": gjr_t_oos_eval["QLIKE"]
}).round(4)

garch_oos_comparison.loc[risk_model_assets]

,GARCH-N MAE,GARCH-t MAE,GJR-t MAE,GARCH-N RMSE,GARCH-t RMSE,GJR-t RMSE,GARCH-N Corr,GARCH-t Corr,GJR-t Corr,GARCH-N QLIKE,GARCH-t QLIKE,GJR-t QLIKE
SPY,0.0565,0.0614,0.0596,0.0932,0.0999,0.0961,0.5436,0.5416,0.5844,0.4354,0.4475,0.4546
EFA,0.0531,0.0527,0.0520,0.0901,0.0899,0.0887,0.4857,0.4834,0.5131,0.3418,0.3455,0.3395
EEM,0.0587,0.0582,0.0583,0.0946,0.0939,0.0954,0.4435,0.4416,0.4565,0.2628,0.2629,0.2721
IEF,0.0149,0.0149,0.0150,0.0224,0.0225,0.0226,0.6136,0.6149,0.6102,0.1969,0.1974,0.1976
TLT,0.0329,0.0329,0.0338,0.0566,0.0573,0.0583,0.4768,0.4764,0.4643,0.2218,0.2226,0.2187
LQD,0.0231,0.0217,0.0213,0.0485,0.0501,0.0493,0.5155,0.4674,0.4779,0.5986,0.7842,0.8370
HYG,0.0277,0.0266,0.0250,0.0467,0.0462,0.0439,0.6845,0.6854,0.7194,0.5126,0.5068,0.4894
GLD,0.0439,0.0434,0.0431,0.0624,0.0627,0.0620,0.5301,0.5266,0.5395,0.2401,0.2436,0.2343
DBC,0.0466,0.0473,0.0486,0.0660,0.0669,0.0688,0.4425,0.4428,0.4335,0.2715,0.2776,0.2890
VNQ,0.0588,0.0590,0.0585,0.1077,0.1093,0.1098,0.5632,0.5586,0.5657,0.3979,0.4187,0.4384


The OOS results should be read asset by asset rather than as a single ranking. The preferred specification varies by metric, and the more flexible models do not dominate consistently across the universe.

This matters for portfolio construction: a model with slightly better in-sample fit is not automatically the best allocation input. The selection should prioritize stable forecasts, acceptable OOS errors, and interpretable behavior across asset classes.


### Out-of-Sample Calibration Regressions


In [185]:
garch_norm_oos_regression = vol_forecast_regression_hac(
    vol_forecast=garch_norm_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

garch_t_oos_regression = vol_forecast_regression_hac(
    vol_forecast=garch_t_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

gjr_t_oos_regression = vol_forecast_regression_hac(
    vol_forecast=gjr_t_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

In [186]:
garch_oos_regression_comparison = pd.DataFrame({
    "GARCH-N Beta": garch_norm_oos_regression["Beta"],
    "GARCH-t Beta": garch_t_oos_regression["Beta"],
    "GJR-t Beta": gjr_t_oos_regression["Beta"],

    "GARCH-N R2": garch_norm_oos_regression["R2"],
    "GARCH-t R2": garch_t_oos_regression["R2"],
    "GJR-t R2": gjr_t_oos_regression["R2"],

    "GARCH-N Joint p-value": garch_norm_oos_regression["Joint p-value"],
    "GARCH-t Joint p-value": garch_t_oos_regression["Joint p-value"],
    "GJR-t Joint p-value": gjr_t_oos_regression["Joint p-value"]
}).round(4)

garch_oos_regression_comparison.loc[risk_model_assets]

,GARCH-N Beta,GARCH-t Beta,GJR-t Beta,GARCH-N R2,GARCH-t R2,GJR-t R2,GARCH-N Joint p-value,GARCH-t Joint p-value,GJR-t Joint p-value
Asset,,,,,,,,,
SPY,0.6467,0.5595,0.5846,0.2955,0.2934,0.3415,0.0027,0.0000,0.0000
EFA,0.5101,0.5099,0.5170,0.2359,0.2336,0.2633,0.0000,0.0000,0.0000
EEM,0.5152,0.5223,0.4916,0.1967,0.1950,0.2084,0.0000,0.0000,0.0000
IEF,0.8350,0.8034,0.7978,0.3765,0.3780,0.3724,0.2039,0.1465,0.1476
TLT,0.7070,0.6658,0.6286,0.2273,0.2269,0.2156,0.1778,0.0740,0.0419
LQD,0.6730,0.6340,0.6708,0.2657,0.2185,0.2284,0.0265,0.0215,0.0344
HYG,0.7280,0.7225,0.7310,0.4685,0.4697,0.5176,0.0002,0.0014,0.0004
GLD,0.7417,0.7093,0.7322,0.2811,0.2773,0.2911,0.0075,0.0065,0.0151
DBC,0.5648,0.5406,0.4996,0.1958,0.1960,0.1879,0.0000,0.0000,0.0000


The OOS calibration regressions show whether forecast levels move with realized volatility after 2018. Positive beta estimates indicate that the forecasts retain timing information. Slopes below one imply compressed forecast variation relative to the realized target, so the models pick up regimes but understate the size of some volatility moves.

The joint calibration tests are stricter than the error metrics and should not be expected to pass uniformly. Rejections mean the forecasts are useful conditional-risk estimates, not statistically unbiased forecasts of the realized-volatility proxy.

Together with the error metrics, this supports a conservative selection: added flexibility is useful only where it improves OOS behavior without creating convergence or stability problems.


## 4. Final Volatility Model Selection

The final volatility input uses Student-t GARCH(1,1) for the risk assets and EWMA/RiskMetrics for SHY.

The rationale is deliberately practical. ARCH-LM tests support conditional-volatility modelling, Student-t innovations improve in-sample fit for the risk assets, and a common symmetric GARCH specification is easier to defend than selecting a different model for every ETF. GJR-GARCH shows asymmetry for some equity- and credit-sensitive assets, but the extra parameter is not used in the final portfolio input because the benefit is uneven and the specification is less stable.

SHY is handled separately because its GARCH estimates show convergence and boundary problems. Using EWMA for the cash proxy avoids forcing an unstable parametric model onto a very low-volatility series.


In [187]:
final_vol_forecast = garch_t_oos_vol.copy().loc[OOS_START:].dropna(how="all")

if CASH_PROXY in final_vol_forecast.columns:
    common_idx = final_vol_forecast.index.intersection(ewma_vol_lagged.index)
    final_vol_forecast.loc[common_idx, CASH_PROXY] = ewma_vol_lagged.loc[
        common_idx,
        CASH_PROXY
    ]

final_vol_forecast = final_vol_forecast.loc[OOS_START:].dropna(how="all")
forecast_sanity_table(final_vol_forecast)


,Min,Median,Mean,Max
SPY,0.0894,0.1526,0.1805,1.0699
EFA,0.0934,0.1522,0.1747,0.9139
EEM,0.1303,0.1928,0.2123,0.9484
IEF,0.0383,0.0609,0.0663,0.1672
TLT,0.0958,0.1385,0.1471,0.4892
LQD,0.0436,0.0632,0.0738,0.4419
HYG,0.0400,0.0618,0.0781,0.5234
GLD,0.1007,0.1518,0.1623,0.4581
DBC,0.0941,0.1654,0.1789,0.4012
VNQ,0.1045,0.1732,0.2015,1.2088


The final forecast distribution is economically plausible. Equities, emerging markets, real estate, commodities, and gold carry higher average volatility, while high-quality fixed income and SHY remain lower-risk. TLT sits above shorter-duration Treasury exposure, consistent with its duration sensitivity.

SHY is handled separately because the Student-t and GJR estimates generated convergence and boundary issues. Using EWMA for SHY avoids forcing a parametric model onto a cash proxy where the GARCH likelihood is unstable.

The final dataset is an ex-ante OOS volatility panel: Student-t GARCH(1,1) for risk assets and EWMA/RiskMetrics for SHY.


In [188]:
final_vol_forecast = final_vol_forecast.loc[OOS_START:].dropna(how="all")
final_vol_forecast.tail()

,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-06-02,0.108341,0.158349,0.273646,0.054106,0.106610,0.055988,0.050196,0.227373,0.225174,0.156776,0.013017
2026-06-03,0.114231,0.158830,0.269169,0.053862,0.105989,0.055849,0.052272,0.225034,0.220899,0.150971,0.012621
2026-06-04,0.111217,0.156697,0.264338,0.053048,0.104686,0.055000,0.051277,0.221862,0.220750,0.168419,0.012382
2026-06-05,0.188150,0.201470,0.396441,0.054960,0.104712,0.058686,0.059674,0.246388,0.228059,0.164694,0.012338
2026-06-08,0.177568,0.194128,0.386696,0.054141,0.104750,0.057515,0.057297,0.241349,0.224612,0.173647,0.014423


In [189]:
# Month-end signal for the portfolio-allocation notebook.
# Use the latest available daily volatility forecast in each month.

final_vol_forecast_monthly = final_vol_forecast.resample("ME").last()
final_vol_forecast_monthly = final_vol_forecast_monthly.loc[OOS_START:].dropna(how="all")
final_vol_forecast_monthly.head()

,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2018-01-31,0.125951,0.116487,0.175975,0.043131,0.111855,0.051887,0.053050,0.113755,0.104167,0.173461,0.005805
2018-02-28,0.217987,0.203929,0.288438,0.050568,0.118593,0.057090,0.074102,0.124334,0.131189,0.234304,0.008656
2018-03-31,0.248786,0.169836,0.280910,0.047364,0.109604,0.051939,0.065390,0.129934,0.125211,0.190325,0.007664
2018-04-30,0.159880,0.109301,0.190073,0.045032,0.108788,0.050354,0.051440,0.121423,0.124299,0.165464,0.006375
2018-05-31,0.140537,0.161440,0.192344,0.056719,0.124478,0.054277,0.052303,0.109455,0.122986,0.141814,0.013591


In [190]:
final_vol_forecast.to_csv("results/volatility_forecast/final_selected_vol_forecast_daily_oos.csv")
final_vol_forecast_monthly.to_csv("results/volatility_forecast/final_selected_vol_forecast_monthly_oos.csv")
